# 长期记忆 · 内存版向量库

**本 Notebook 目标：** 演示如何把用户偏好、事实等持久化存储，并在后续对话中按语义检索召回，弥补滑动窗口只能保留近期消息的局限。

> 嵌入：SentenceTransformer（`BAAI/bge-small-zh-v1.5`）| 对话：DeepSeek / OpenAI | 存储：内存 Dict（可替换为 FAISS / Milvus / Qdrant / pgvector）



## 整体架构

```mermaid
%%{init: {'flowchart': {'curve': 'linear'}}}%%
flowchart TB
    subgraph Store["SimpleLongTermMemoryStore"]
        DICT[("items: Dict[id, MemoryItem]")]
        EMB["Embedder\nBAAI/bge-small-zh-v1.5"]
    end

    ADD["add(user_id, text, importance)"] --> EMB
    EMB --> ITEM["MemoryItem\nid / user_id / text / vector / importance / 时间戳"]
    ITEM --> DICT

    SEARCH["search(user_id, query, top_k)"] --> EMB2["embed_one(query)"]
    EMB2 --> SCORE["综合打分"]
    DICT --> SCORE
    SCORE --> TOP["返回 top_k 条记忆"]
```

## 生产替换建议

| 组件 | 当前实现 | 可替换为 |
|------|----------|----------|
| 嵌入 | 本地 SentenceTransformer | OpenAI / DeepSeek Embedding API |
| 对话 | OpenAI SDK（DeepSeek 兼容） | 任意 Chat Completions API |
| 存储 | 内存 `Dict` | FAISS、Milvus、Qdrant、pgvector |
| 检索 | 暴力遍历 + 综合打分 | ANN 索引 + 后过滤 / 重排 |

---
# Part 1 · 环境准备

自动检测并安装 `numpy` / `openai` / `python-dotenv`；加载项目根目录 `.env`。

> 默认 `USE_MOCK_EMBEDDER=1`（字符 n-gram Mock 嵌入，无需 `sentence-transformers`）。若本地已装好模型，可设 `USE_MOCK_EMBEDDER=0` 启用 `BAAI/bge-small-zh-v1.5`。

In [1]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

try:
    sys.stdout.reconfigure(encoding="utf-8")
except (AttributeError, OSError):
    pass  # Jupyter OutStream 不支持 reconfigure


def _ensure_package(import_name: str, pip_name: str) -> None:
    try:
        importlib.import_module(import_name)
    except ImportError:
        print(f"[setup] 安装 {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])


for _import, _pip in [
    ("numpy", "numpy"),
    ("dotenv", "python-dotenv"),
    ("openai", "openai"),
]:
    _ensure_package(_import, _pip)

from dotenv import load_dotenv

for env_path in [Path.cwd() / ".env", Path.cwd().parent / ".env"]:
    if env_path.exists():
        load_dotenv(env_path)
        print(f"[setup] 已加载 {env_path}")
        break
else:
    load_dotenv()

# True = 强制 Mock 嵌入（无需 sentence-transformers）；False = 优先真实模型
USE_MOCK_EMBEDDER = os.environ.get("USE_MOCK_EMBEDDER", "1") == "1"

_NOTEBOOK_READY = False  # Part 3 完成后置 True
print(f"[setup] 就绪 | Python {sys.version.split()[0]} | mock_embedder={USE_MOCK_EMBEDDER}")

[setup] 已加载 d:\workspace\doc\面试狂魔\人工智能面试题\AI_coding_interview\.env
[setup] 就绪 | Python 3.11.9 | mock_embedder=True


# Part 2 · 数据模型与嵌入

In [2]:
import hashlib
import math
import time
from dataclasses import dataclass
from typing import Dict, List, Optional, Protocol

import numpy as np


@dataclass
class MemoryItem:
    id: str
    user_id: str
    text: str
    vector: List[float]
    importance: float  # 0~1
    created_at: float
    last_accessed: float


class EmbedderProtocol(Protocol):
    def embed_one(self, text: str) -> List[float]: ...


class SentenceTransformerEmbedder:
    """真实语义嵌入；归一化后余弦相似度 = 点积。"""

    def __init__(self, model: str = "BAAI/bge-small-zh-v1.5", batch_size: int = 32):
        from sentence_transformers import SentenceTransformer

        self.batch_size = batch_size
        print(f"[Embedder] 加载本地模型：{model}")
        self._model = SentenceTransformer(model)

    def embed(self, texts: List[str]) -> List[List[float]]:
        all_embeddings: List[List[float]] = []
        for i in range(0, len(texts), self.batch_size):
            batch = texts[i : i + self.batch_size]
            vecs = self._model.encode(batch, normalize_embeddings=True).astype("float32")
            all_embeddings.extend(vecs.tolist())
        return all_embeddings

    def embed_one(self, text: str) -> List[float]:
        return self.embed([text])[0]


class MockEmbedder:
    """轻量嵌入回退：字符 n-gram 哈希 → L2 归一化（无需 sentence-transformers）。"""

    DIM = 512

    @staticmethod
    def _ngrams(text: str, n: int) -> List[str]:
        s = text.strip()
        return [s[i : i + n] for i in range(max(0, len(s) - n + 1))]

    def _to_vec(self, text: str) -> np.ndarray:
        vec = np.zeros(self.DIM, dtype=np.float32)
        tokens = list(text.strip()) + self._ngrams(text, 2) + self._ngrams(text, 3)
        for tok in tokens:
            vec[hash(tok) % self.DIM] += 1.0
        norm = float(np.linalg.norm(vec))
        return vec / norm if norm > 0 else vec

    def embed(self, texts: List[str]) -> List[List[float]]:
        return [self._to_vec(t).tolist() for t in texts]

    def embed_one(self, text: str) -> List[float]:
        return self._to_vec(text).tolist()


def create_embedder(
    model: str = "BAAI/bge-small-zh-v1.5",
    use_mock: Optional[bool] = None,
) -> EmbedderProtocol:
    if use_mock is None:
        use_mock = globals().get("USE_MOCK_EMBEDDER", False)
    if use_mock:
        print("[Embedder] Mock 模式（无需下载模型）")
        return MockEmbedder()
    try:
        return SentenceTransformerEmbedder(model=model)
    except Exception as exc:
        print(f"[Embedder] SentenceTransformer 不可用 ({exc.__class__.__name__})，回退 Mock")
        return MockEmbedder()


def cosine_sim(a: List[float], b: List[float]) -> float:
    return float(np.dot(a, b))

print("[Part 2] MemoryItem / Embedder 已定义")

[Part 2] MemoryItem / Embedder 已定义


# Part 3 · 长期记忆存储

In [3]:
class SimpleLongTermMemoryStore:
    """内存版长期记忆存储（生产可替换为 FAISS、Milvus、Qdrant、pgvector 等）。"""

    def __init__(
        self,
        embedder: Optional[EmbedderProtocol] = None,
        model: str = "BAAI/bge-small-zh-v1.5",
    ):
        self.embedder = embedder or create_embedder(model=model)
        self.items: Dict[str, MemoryItem] = {}

    def add(self, user_id: str, text: str, importance: float = 0.5) -> str:
        mem_id = hashlib.md5(
            f"{user_id}:{text}:{time.time()}".encode("utf-8")
        ).hexdigest()[:12]
        now = time.time()
        vec = self.embedder.embed_one(text)
        self.items[mem_id] = MemoryItem(
            id=mem_id,
            user_id=user_id,
            text=text,
            vector=vec,
            importance=max(0.0, min(1.0, importance)),
            created_at=now,
            last_accessed=now,
        )
        return mem_id

    def delete(self, mem_id: str) -> None:
        self.items.pop(mem_id, None)

    def update_text(self, mem_id: str, new_text: str) -> None:
        if mem_id not in self.items:
            return
        it = self.items[mem_id]
        self.items[mem_id] = MemoryItem(
            id=it.id,
            user_id=it.user_id,
            text=new_text,
            vector=self.embedder.embed_one(new_text),
            importance=it.importance,
            created_at=it.created_at,
            last_accessed=time.time(),
        )

    def search(
        self,
        user_id: str,
        query: str,
        top_k: int = 5,
        decay_lambda: float = 2e-7,
    ) -> List[tuple[float, MemoryItem]]:
        q = self.embedder.embed_one(query)
        now = time.time()
        scored: List[tuple[float, MemoryItem]] = []
        for it in self.items.values():
            if it.user_id != user_id:
                continue
            rel = cosine_sim(q, it.vector)
            age_sec = max(0.0, now - it.last_accessed)
            recency = math.exp(-decay_lambda * age_sec)
            score = 0.6 * rel + 0.2 * it.importance + 0.2 * recency
            scored.append((score, it))
        scored.sort(key=lambda x: x[0], reverse=True)
        return scored[:top_k]

    def explain_search(
        self,
        user_id: str,
        query: str,
        top_k: int = 5,
        decay_lambda: float = 2e-7,
    ) -> None:
        """打印检索结果及三项得分拆解，方便课堂讲解。"""
        q = self.embedder.embed_one(query)
        now = time.time()
        rows = []
        for it in self.items.values():
            if it.user_id != user_id:
                continue
            rel = cosine_sim(q, it.vector)
            age_sec = max(0.0, now - it.last_accessed)
            recency = math.exp(-decay_lambda * age_sec)
            score = 0.6 * rel + 0.2 * it.importance + 0.2 * recency
            rows.append((score, rel, it.importance, recency, it.text))
        rows.sort(key=lambda x: x[0], reverse=True)

        print(f"Query: {query}")
        print(f"{'#':<3} {'score':>7} {'rel':>7} {'imp':>5} {'rec':>7}  text")
        print("-" * 72)
        for i, (score, rel, imp, rec, text) in enumerate(rows[:top_k], 1):
            print(f"{i:<3} {score:>7.4f} {rel:>7.4f} {imp:>5.2f} {rec:>7.4f}  {text}")


_NOTEBOOK_READY = True
print("[Part 3] SimpleLongTermMemoryStore 已定义，Notebook 可演示")

[Part 3] SimpleLongTermMemoryStore 已定义，Notebook 可演示


---
# Part 4 · 运行示例

分四步演示：**写入** → **更新** → **检索（得分拆解）** → **用户隔离**。

> **Run All** 一键运行；若单独运行本 Cell，需先运行 Part 1–3。

In [4]:
def _require_notebook_ready() -> None:
    if not globals().get("_NOTEBOOK_READY", False):
        raise RuntimeError(
            "请先 Run All，或依次运行 Part 1 → Part 2 → Part 3 的代码 Cell"
        )


_require_notebook_ready()
store = SimpleLongTermMemoryStore()

print("=== Step 1 · 写入两条记忆 ===")
mid = store.add("u1", "用户偏好：沟通风格简洁。", importance=0.9)
store.add("u1", "用户住在上海，偏好早上开会。", importance=0.6)
for item in store.items.values():
    print(f"  [{item.id}] imp={item.importance} | {item.text}")

print("\n=== Step 2 · 更新第一条记忆 ===")
store.update_text(mid, "用户偏好：沟通风格简洁；禁用表情。")
print(f"  更新后: {store.items[mid].text}")

print("\n=== Step 3 · 语义检索 + 得分拆解 ===")
store.explain_search("u1", "他喜欢怎么说话？")

print("\n=== Step 4 · 用户隔离（u2 看不到 u1 的记忆）===")
store.add("u2", "用户住在北京，偏好晚上开会。", importance=0.5)
hits_u2 = store.search("u2", "他喜欢怎么说话？")
print(f"  u2 检索命中 {len(hits_u2)} 条（仅 u2 自己的记忆）")
for score, item in hits_u2:
    print(f"  {score:.4f} | {item.text}")

hits_u1 = store.search("u1", "他喜欢怎么说话？")
assert len(hits_u1) >= 2, "u1 应至少检索到 2 条记忆"
assert all(item.user_id == "u1" for _, item in hits_u1), "u1 检索结果应全部属于 u1"
assert len(hits_u2) == 1 and hits_u2[0][1].user_id == "u2", "u2 只能看到自己的记忆"
assert "偏好" in hits_u1[0][1].text, "Top-1 应为偏好相关记忆"
print("\n[check] 向量库演示通过 ✓")

[Embedder] Mock 模式（无需下载模型）
=== Step 1 · 写入两条记忆 ===
  [62d304ceb494] imp=0.9 | 用户偏好：沟通风格简洁。
  [e204c7cf5df9] imp=0.6 | 用户住在上海，偏好早上开会。

=== Step 2 · 更新第一条记忆 ===
  更新后: 用户偏好：沟通风格简洁；禁用表情。

=== Step 3 · 语义检索 + 得分拆解 ===
Query: 他喜欢怎么说话？
#     score     rel   imp     rec  text
------------------------------------------------------------------------
1    0.4129  0.0548  0.90  1.0000  用户偏好：沟通风格简洁；禁用表情。
2    0.3391  0.0318  0.60  1.0000  用户住在上海，偏好早上开会。

=== Step 4 · 用户隔离（u2 看不到 u1 的记忆）===
  u2 检索命中 1 条（仅 u2 自己的记忆）
  0.3191 | 用户住在北京，偏好晚上开会。

[check] 向量库演示通过 ✓


---
# Part 5 · 真实 LLM 对话 + 长期记忆

流程：**对话 → LLM 提取事实写入向量库 → 后续对话语义检索召回 → 注入 Prompt 个性化回复**。

> 需配置 `DEEPSEEK_API_KEY` 或 `OPENAI_API_KEY`（项目根目录 `.env`）。

In [ ]:
from typing import Optional

from openai import OpenAI

def create_client() -> tuple[Optional[OpenAI], str]:
     """DeepSeek"""
     deepseek_key = "sk-your-api-key-here"
     print("[LLM] DeepSeek deepseek-chat")
     return OpenAI(api_key=deepseek_key, base_url="https://api.deepseek.com"), "deepseek-chat"


client, llm_model = create_client()

[LLM] DeepSeek deepseek-chat


In [10]:
def llm_extract_memories(
    client: OpenAI,
    model: str,
    user_text: str,
    assistant_text: str,
) -> list[tuple[str, float]]:
    """LLM 从一轮对话中提取可持久化的事实/偏好。"""
    prompt = f"""分析下面一轮对话，提取关于用户的、值得长期记住的事实或偏好。
每行一条，格式简短，如：用户偏好：沟通风格简洁
若没有可记住的内容，只输出 NONE。

用户：{user_text}
助手：{assistant_text}"""
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    text = (response.choices[0].message.content or "").strip()
    if not text or text.upper() == "NONE":
        return []
    lines = [
        ln.strip().lstrip("-• ")
        for ln in text.splitlines()
        if ln.strip() and ln.strip().upper() != "NONE"
    ]
    return [(ln, 0.8) for ln in lines]


def _print_messages(messages: list[dict]) -> None:
    """格式化打印发给 LLM 的 messages。"""
    print("\n[LLM request]")
    for msg in messages:
        role = msg["role"]
        content = msg["content"]
        bar = "─" * 52
        print(f"  ┌─ {role} " + "─" * max(0, 44 - len(role)))
        for line in content.splitlines() or [""]:
            print(f"  │ {line}")
        print(f"  └{bar}")


def chat_with_long_term_memory(
    store: SimpleLongTermMemoryStore,
    user_id: str,
    client: OpenAI,
    model: str,
    user_text: str,
) -> str:
    """检索长期记忆 → 注入 system prompt → 真实 LLM 回复 → 提取新记忆写入。"""
    hits = store.search(user_id, user_text, top_k=3)
    if hits:
        memory_block = "\n".join(
            f"- {item.text} (score={score:.3f})" for score, item in hits
        )
        print(f"  [memory recall] 命中 {len(hits)} 条")
    else:
        memory_block = "（暂无）"

    messages = [
        {
            "role": "system",
            "content": (
                "你是助手，回答尽量简洁。"
                f"以下是该用户的长期记忆，请据此个性化回复：\n{memory_block}"
            ),
        },
        {"role": "user", "content": user_text},
    ]

    _print_messages(messages)
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.7,
    )
    answer = (response.choices[0].message.content or "").strip()

    for text, importance in llm_extract_memories(client, model, user_text, answer):
        mem_id = store.add(user_id, text, importance=importance)
        print(f"  [memory+] {text}  (id={mem_id})")
    return answer

In [12]:
if client is None:
    print("[skip] 未配置 DEEPSEEK_API_KEY / OPENAI_API_KEY，跳过 Part 5")
else:
    try:
        chat_store = SimpleLongTermMemoryStore()
        USER_ID = "demo_user"

        print("=== 第一轮对话 · LLM 提取并写入长期记忆 ===")
        session1_turns = [
            "我叫阿明，回复请简洁，不要用表情符号。",
            "我住在上海，开会尽量安排在上午。",
        ]
        for user_msg in session1_turns:
            print(f"[user] {user_msg}")
            reply = chat_with_long_term_memory(chat_store, USER_ID, client, llm_model, user_msg)
            print(f"[assistant] {reply}\n")

        print("=== 语义检索（当前记忆库）===")
        chat_store.explain_search(USER_ID, "用户的沟通习惯和所在地")

        print("\n=== 第二轮对话 · 跨轮召回记忆 ===")
        follow_up = "帮我写一段简短的站会开场白，按我的习惯来。"
        print(f"[user] {follow_up}")
        reply2 = chat_with_long_term_memory(chat_store, USER_ID, client, llm_model, follow_up)
        print(f"[assistant] {reply2}\n")

        print("=== 第三轮对话 · 综合召回多条记忆 ===")
        follow_up2 = "下周我在上海有个客户演示，帮我写三句简洁的开场白。"
        print(f"[user] {follow_up2}")
        reply3 = chat_with_long_term_memory(chat_store, USER_ID, client, llm_model, follow_up2)
        print(f"[assistant] {reply3}")

        hits = chat_store.search(USER_ID, "沟通风格", top_k=1)
        assert hits, "应检索到至少一条记忆"
        print("\n[check] LLM + 长期记忆演示通过 ✓")
    except Exception as exc:
        print(f"[skip] Part 5 调用失败 ({exc.__class__.__name__}: {exc})")
        print("  请检查 .env 中的 DEEPSEEK_API_KEY / OPENAI_API_KEY 是否有效")

[Embedder] Mock 模式（无需下载模型）
=== 第一轮对话 · LLM 提取并写入长期记忆 ===
[user] 我叫阿明，回复请简洁，不要用表情符号。

[LLM request]
  ┌─ system ──────────────────────────────────────
  │ 你是助手，回答尽量简洁。以下是该用户的长期记忆，请据此个性化回复：
  │ （暂无）
  └────────────────────────────────────────────────────
  ┌─ user ────────────────────────────────────────
  │ 我叫阿明，回复请简洁，不要用表情符号。
  └────────────────────────────────────────────────────
  [memory+] 用户姓名：阿明  (id=96c25f239996)
  [memory+] 用户偏好：回复简洁  (id=8a495631fa24)
  [memory+] 用户偏好：不使用表情符号  (id=a9a072b70f2d)
[assistant] 明白，阿明。请说。

[user] 我住在上海，开会尽量安排在上午。
  [memory recall] 命中 3 条

[LLM request]
  ┌─ system ──────────────────────────────────────
  │ 你是助手，回答尽量简洁。以下是该用户的长期记忆，请据此个性化回复：
  │ - 用户偏好：回复简洁 (score=0.391)
  │ - 用户姓名：阿明 (score=0.379)
  │ - 用户偏好：不使用表情符号 (score=0.374)
  └────────────────────────────────────────────────────
  ┌─ user ────────────────────────────────────────
  │ 我住在上海，开会尽量安排在上午。
  └────────────────────────────────────────────────────
  [memory+] 用户偏好：会议安排在上午  (id=fc43b2348ca